In [ ]:
!pip install -q -U langchain_groq langchain_community pypdf sentence-transformers chromadb

In [ ]:
!pip install -q -U langchain-text-splitters

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq


loader = PyPDFLoader("PDF name.pdf")
data = loader.load()


text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(data)


embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)

print(f"Successfully split into {len(chunks)} chunks and stored in Chroma.")

In [ ]:
!pip install -q -U langchain-classic

In [ ]:
from langchain_classic.chains import RetrievalQA
from langchain_groq import ChatGroq
import getpass
import os


os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0
)


qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever()
)


query = "Can you give me a 3-bullet point summary of this PDF?"
response = qa_chain.invoke(query)

print("--- ANSWER ---")
print(response["result"])